# Sentiment analysis with an MLP and BOW representation

## Imports

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import time

In [ ]:
from sklearn import set_config
set_config(display="diagram")

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [ ]:
import nltk
from nltk import word_tokenize, sent_tokenize    
from nltk.stem import PorterStemmer
from nltk import FreqDist

In [ ]:
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras.models import Model
from tensorflow.keras import layers, callbacks, utils

## Load the dataset

* **Training Dataset:** The sample of data used to fit the model.
* **Validation Dataset:** The sample of data used to provide an unbiased evaluation of a model fit on the training dataset while tuning model hyperparameters. The evaluation becomes more biased as skill on the validation dataset is incorporated into the model configuration.
* **Test Dataset:** The sample of data used to provide an unbiased evaluation of a final model fit on the training dataset.

In [ ]:
TRAIN = pd.read_csv("http://www.i3s.unice.fr/~riveill/dataset/Amazon_Unlocked_Mobile/train.csv.gz")
VAL = pd.read_csv("http://www.i3s.unice.fr/~riveill/dataset/Amazon_Unlocked_Mobile/val.csv.gz")
TEST = pd.read_csv("http://www.i3s.unice.fr/~riveill/dataset/Amazon_Unlocked_Mobile/test.csv.gz")

TRAIN.head()

As I'm going to be doing cross validation, I'm going to merge the train part and the val part.

In [ ]:
# I chose to merge TRAIN and VAL
TRAIN = pd.concat([TRAIN, VAL], axis=0)
TRAIN.shape, TEST.shape

In [ ]:
# I choose to replace missing review by empty one
TRAIN = TRAIN.fillna("")
TEST = TEST.fillna("")

## Build X (features vectors) and y (labels)

In [ ]:
# Construct X_train and y_train
X_train = np.array(TRAIN['Reviews']).reshape(-1,1)
y_train = np.array(TRAIN['Rating']-np.min(TRAIN['Rating'])).reshape(-1,1)
X_train.shape, y_train.shape

In [ ]:
# Construct X_test and y_test
X_test = np.array(TEST['Reviews']).reshape(-1,1)
y_test = np.array(TEST['Rating']-np.min(TRAIN['Rating'])).reshape(-1,1)
X_test.shape, y_test.shape

In [ ]:
assert np.min(y_train)==0
assert np.min(y_test)==0

In [ ]:
del TRAIN, VAL, TEST

## Very small EDA (Exploratory Data Analysis)

To choose certain constants (size of vocabulary, length of a line, etc.), it is good to know the dataset used.

You can also add new features to the dataset.

In [ ]:
# What is the vocabulary size ?

# Tokenized the reviews
reviews_tokenized = [word_tokenize(review) for review in X_train.ravel()]

# Count the vocabulary
flatten_reviews = [item for sublist in reviews_tokenized for item in sublist]
vocabulary_size = len(set(flatten_reviews))
vocabulary_size

In [ ]:
# Initialize max_length with 90 % of the review not truncated
from nltk import FreqDist

n = 0.90 # 90 % of reviews are not truncated
lengths = [len(txt.split()) for txt in X_train.ravel()]
lengths_fdist = FreqDist(lengths) 
first_25 = FreqDist(dict(lengths_fdist.most_common()[:25]))
first_25.plot()

cumul = 0
for length in range(max(lengths)):
    cumul += lengths_fdist[length]
    if cumul>=n*len(X_train):
        break

max_len = length
max_len # With this length, 90% of reviews are not truncated

In [ ]:
# Intialize vocab_size in order to keep words that frequency is more than 1 %% of the sentence
from nltk import FreqDist

fdist = FreqDist([w for txt in X_train.ravel() for w in txt.split()])  
first_25 = FreqDist(dict(fdist.most_common()[:25]))
first_25.plot()

threshold = 0.001 * len(X_train) # We keep all the words that appear in at least 1 %% of the number of documents

for i, (word, nb) in enumerate(fdist.most_common()):
    if nb<threshold:
        break

max_tokens = i
max_tokens # Only words present in 1 %% of reviews are retained

In [ ]:
# Is the dataset is balanced ?

pd.Series(y_train.ravel()).value_counts()

Other EDA ideas are described in the following papers :
* [Fundamental EDA Techniques for NLP](https://towardsdatascience.com/fundamental-eda-techniques-for-nlp-f81a93696a75)
* [Intermediate EDA Techniques for NLP](https://towardsdatascience.com/intermediate-eda-techniques-for-nlp-2c898cc96d1d)

## Baseline model

In [ ]:
# I define the pipeline
lr_pipeline = Pipeline([
        ('feature_extraction',  CountVectorizer()),
        ('classification',  LogisticRegression(multi_class='auto', max_iter=400))
        ])

display(lr_pipeline)

# I fit the model
lr_pipeline.fit(X_train.ravel(), y_train)

# I evaluate the model
y_pred = lr_pipeline.predict(X_test.ravel())

print(classification_report(y_pred, y_test))

## Build an MLP Classifier

In [ ]:
# Define constant
nb_features = X_train.shape[1:]
num_classes = len(np.unique(y_train))
dropout_rate = 0.6
hidden_size = [1024,32]
hidden_act = "relu"

nb_features

In [ ]:
# Create the layer.
# Build the vectorizer
vectorize_layer = tf.keras.layers.TextVectorization(max_tokens=max_tokens, # or None
                                                    standardize='lower_and_strip_punctuation',
                                                    split='whitespace',
                                                    ngrams=1,
                                                    output_mode='count',
                                                    pad_to_max_tokens=True,
                                                    vocabulary=None)
vectorize_layer.adapt(X_train)

# Now that the vocab layer has been created, call `adapt` on the
# text-only dataset to create the vocabulary. You don't have to batch,
# but for large datasets this means we're not keeping spare copies of
# the dataset.
vectorize_layer.adapt(X_train)

In [ ]:
vocabulary = vectorize_layer.get_vocabulary()

assert len(vocabulary)==max_tokens

vocabulary[:10]

In [ ]:
# Create the model that uses the vectorize text layer

# Set the input layer
# It needs to have a shape of (1,) (because we need to guarantee that there is exactly
# one string input per batch), and the dtype needs to be 'string'.
text_input = layers.Input(shape=(nb_features), dtype=tf.string, name='input')

# The first layer in our model is the vectorization layer.
h =vectorize_layer(text_input)

# Set the hidden layer
for i, s in enumerate(hidden_size):
    h = layers.Dense(s, activation=hidden_act, name='hidden'+str(i))(h)
    h = layers.Dropout(dropout_rate)(h)

# Set the output layer
outputs = layers.Dense(num_classes, activation='softmax', name='output')(h)

# Build the model
model = Model(text_input, outputs)

# Print the model
model.summary()

In [ ]:
utils.plot_model(model, show_shapes=True,
    show_dtype=True,
    show_layer_names=True,
    rankdir="TB",
    expand_nested=True,
    dpi=96,
    show_layer_activations=True,
    show_trainable=True,
                )

In [ ]:
# Compile the model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# Stop training with early stopping with patience of 5
callbacks_list = [callbacks.EarlyStopping(monitor='val_accuracy', mode='max',
                                          min_delta=0.0005,  patience=5, 
                                          restore_best_weights=True,
                                          verbose=1),
                                          ]

history = model.fit(X_train, y_train,
                    validation_split=0.2,
                    epochs=1000, batch_size=256,
                    callbacks=callbacks_list,
                    verbose=1)

In [ ]:
# Plot the learning curves and analyze them
plt.plot(history.history['loss'], label="loss")
plt.plot(history.history['val_loss'], label="val_loss")
plt.show()

In [ ]:
# Evaluate the model
y_pred_encoded = model.predict(X_test)
y_pred = np.argmax(y_pred_encoded,axis=1)

print(classification_report(y_test, y_pred))

<pre>
              precision    recall  f1-score   support

           0       0.64      0.75      0.69       159
           1       0.40      0.12      0.18        51
           2       0.32      0.10      0.15        71
           3       0.46      0.15      0.22       163
           4       0.73      0.96      0.83       556

    accuracy                           0.69      1000
   macro avg       0.51      0.41      0.41      1000
weighted avg       0.63      0.69      0.63      1000
</pre>